# 01 — Inspección inicial

**Dataset:** `streaming_users_dirty.json` — usuarios de una plataforma de streaming en Latinoamérica.
Cada registro representa **un usuario** con su plan de suscripción, país, edad, consumo mensual (minutos),
género favorito, fecha de último inicio de sesión y cantidad de tickets de soporte.

**Objetivo de este notebook:** reunir evidencia sobre estructura, tipos, faltantes, duplicados e
inconsistencias, sin tomar decisiones definitivas. Esta evidencia justifica las acciones del notebook
`02_calidad_y_limpieza.ipynb`.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

df = pd.read_json("../data/raw/streaming_users_dirty.json")
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")

Dimensiones: 8160 filas x 8 columnas


## Estructura general y tipos de datos

In [2]:
df.head()

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   object 
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   object 
 5   favorite_genre            7920 non-null   object 
 6   last_login_date           7840 non-null   object 
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), object(4)
memory usage: 510.1+ KB


**Observación:** `last_login_date` llega como texto (`object`) y no como fecha: habrá que validar su formato.
El resto de los tipos es consistente con lo esperado (numéricos y categóricos de texto).

## Valores faltantes

In [4]:
faltantes = pd.DataFrame({
    "nulos": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(2)
})
faltantes

,nulos,porcentaje
user_id,0,0.00
age,0,0.00
subscription_plan,0,0.00
monthly_watch_time_mins,193,2.37
country,0,0.00
favorite_genre,240,2.94
last_login_date,320,3.92
customer_support_tickets,0,0.00


**Evidencia:** hay faltantes en `monthly_watch_time_mins` (~2,4 %), `favorite_genre` (~2,9 %) y
`last_login_date` (~3,9 %). Ninguna columna supera el 4 %, por lo que no se anticipa eliminar columnas.

## Duplicados

In [5]:
print("Filas duplicadas exactas:", df.duplicated().sum())
print("user_id duplicados:", df["user_id"].duplicated().sum())

Filas duplicadas exactas: 126
user_id duplicados: 160


**Evidencia:** existen 126 filas duplicadas exactas y 160 `user_id` repetidos. Como cada registro debe
representar un usuario único, los repetidos por `user_id` (aun con alguna diferencia menor entre copias)
también deberán revisarse en la etapa de limpieza.

## Estadísticos de las variables numéricas

In [6]:
df.describe().round(2)

,user_id,age,monthly_watch_time_mins,customer_support_tickets
count,8160.00,8160.00,7967.00,8160.00
mean,13995.43,34.10,1107.35,1.80
std,2310.81,14.51,5310.44,11.33
min,10000.00,-5.00,-120.00,-1.00
25%,11987.75,25.00,489.20,0.00
50%,13998.50,33.00,757.40,1.00
75%,15997.25,42.00,1045.70,1.00
max,17999.00,150.00,99999.00,150.00


**Evidencia de valores imposibles:**
- `age`: mínimo **-5** y máximo **150** (fuera de todo rango humano razonable).
- `monthly_watch_time_mins`: mínimo **-120** (un tiempo no puede ser negativo) y máximo **99999**,
  muy alejado del percentil 75 (~1046): parece un valor centinela de "sin dato".
- `customer_support_tickets`: mínimo **-1** (posible centinela) y máximo **150**, incompatible con
  el resto de la distribución (mediana 1).

In [7]:
print("age fuera de [13, 100]:", ((df["age"] < 13) | (df["age"] > 100)).sum())
print("watch_time negativos:", (df["monthly_watch_time_mins"] < 0).sum())
print("watch_time >= 10000 (centinela 99999):", (df["monthly_watch_time_mins"] >= 10000).sum())
print("tickets negativos:", (df["customer_support_tickets"] < 0).sum())
print("tickets > 30:", (df["customer_support_tickets"] > 30).sum(),
      "| valores:", sorted(df.loc[df["customer_support_tickets"] > 30, "customer_support_tickets"].unique()))

age fuera de [13, 100]: 120
watch_time negativos: 49
watch_time >= 10000 (centinela 99999): 31
tickets negativos: 29
tickets > 30: 67 | valores: [np.int64(99), np.int64(150)]


## Variables categóricas: inconsistencias de escritura

In [8]:
for col in ["subscription_plan", "country", "favorite_genre"]:
    valores = sorted(df[col].dropna().unique().tolist())
    print(f"--- {col}: {len(valores)} valores distintos ---")
    print(valores, chr(10))

--- subscription_plan: 15 valores distintos ---
['BASICO', 'Basic', 'Básico', 'Estándar', 'Estándar ', 'PREMIUM', 'Premium', 'Premium ', 'Premiun', 'STANDARD', 'Std', 'basico', 'básico', 'estandar', 'premium'] 

--- country: 26 valores distintos ---
['ARG', 'Argentina', 'Argentina ', 'BRA', 'Brasil', 'Brazil', 'CHL', 'COL', 'Chile', 'Chile ', 'Colombia', 'MEX', 'Mexico', 'México', 'PER', 'Peru', 'Perú', 'URY', 'Uruguay', 'argentina', 'brasil', 'chile', 'colombia', 'méxico', 'perú', 'uruguay'] 

--- favorite_genre: 28 valores distintos ---
['ACCIÓN', 'Acción', 'Action', 'COMEDIA', 'CRIME', 'Comedia', 'Comedia ', 'Crime', 'Crimen', 'DOC', 'DRAMA', 'Documental', 'Documentary', 'Drama', 'Drama ', 'ROMANCE', 'Romance', 'Romance ', 'THRILLER', 'Thriller', 'Thriller ', 'accion', 'comedy', 'crime', 'documental', 'drama', 'romance', 'thriler'] 



**Evidencia:** las tres columnas mezclan mayúsculas/minúsculas, idiomas (Brasil/Brazil, Crimen/Crime),
códigos ISO (ARG, MEX), abreviaturas (Std, DOC), errores de tipeo (Premiun, thriler) y espacios finales
("Chile "). Conceptualmente hay solo **3 planes**, **7 países** y **7 géneros**.

## Fechas de último inicio de sesión

In [9]:
s = df["last_login_date"].astype("string")
d_iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
d_lat = pd.to_datetime(s, format="%d-%m-%Y", errors="coerce")
print("No parseables como AAAA-MM-DD (sin contar nulos):", int(d_iso.isna().sum() - s.isna().sum()))
print("Recuperables con formato DD-MM-AAAA:", int((d_iso.isna() & d_lat.notna()).sum()))
combinadas = d_iso.fillna(d_lat)
print("Inválidas en ambos formatos (sin contar nulos):", int(combinadas.isna().sum() - s.isna().sum()))
print("Fechas futuras (> 2026-07-04):", int((combinadas > pd.Timestamp("2026-07-04")).sum()))
print("Ejemplos de valores problemáticos:",
      [v for v in ["0000-00-00", "not_available", "2026-15-03", "31-02-2022", "2029-01-01"] if (s == v).any()])

No parseables como AAAA-MM-DD (sin contar nulos): 449
Recuperables con formato DD-MM-AAAA: 174
Inválidas en ambos formatos (sin contar nulos): 275
Fechas futuras (> 2026-07-04): 15
Ejemplos de valores problemáticos: ['0000-00-00', 'not_available', '2026-15-03', '31-02-2022', '2029-01-01']


## Observaciones iniciales (síntesis de evidencia)

1. **8.160 filas y 8 columnas**; cada fila es un usuario.
2. **Duplicados:** 126 filas exactas y 160 `user_id` repetidos.
3. **Faltantes** en 3 columnas, todas por debajo del 4 %.
4. **Valores imposibles** en las tres numéricas de comportamiento (edades negativas/150,
   tiempos negativos y centinela 99999, tickets -1/99/150).
5. **Categóricas inconsistentes:** 15 variantes para 3 planes, 26 para 7 países, 28 para 7 géneros.
6. **Fechas:** dos formatos mezclados, valores inválidos (`0000-00-00`, `31-02-2022`, `not_available`)
   y fechas futuras imposibles (`2029-01-01`).

## Preguntas de análisis

- **P1.** ¿Cómo se distribuye el tiempo de visualización mensual de los usuarios?
- **P2.** ¿Difiere el consumo (minutos mensuales) entre planes de suscripción?
- **P3.** ¿Existe relación entre la edad del usuario y su tiempo de visualización?
- **P4.** ¿Las preferencias de género varían según el país?
- **P5.** ¿Pueden resumirse las variables numéricas de comportamiento en pocas componentes principales?